# 01 — Data Quality and Exploratory Data Analysis

## Objective
Inspect the Sample Superstore transaction file before cleaning or modelling. Establish its grain, schema, coverage, completeness, uniqueness and basic commercial profile.

**Input:** `data/raw/Sample - Superstore.csv` (downloaded separately; do not commit unless redistribution rights are confirmed).

**Encoding note:** This copy contains Windows-1252 characters, so it is loaded with `encoding='cp1252'`.

**Important scope note:** This dataset has region, state and city, but no store identifier or inventory-on-hand field. It cannot by itself support store-level performance, stock cover, stockouts or inventory turnover.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from IPython.display import display

DATA_PATH = Path('../data/raw/Sample - Superstore.csv')
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f'Could not find {DATA_PATH}. Download the source CSV and place it in data/raw/.'
    )

df = pd.read_csv(DATA_PATH, encoding='cp1252', low_memory=False)
print(f'Rows: {df.shape[0]:,} | Columns: {df.shape[1]:,}')
display(df.head())

## 1. Schema and grain

Each row appears to represent an order line. We will validate the row identifier and compare transaction-line count with distinct order count.

In [ ]:
schema = pd.DataFrame({
    'column': df.columns,
    'dtype': df.dtypes.astype(str).values,
    'missing_count': df.isna().sum().values,
    'missing_pct': (df.isna().mean().values * 100).round(2)
})
display(schema)

print('Row ID unique:', df['Row ID'].is_unique)
print('Distinct orders:', df['Order ID'].nunique())
print('Distinct customers:', df['Customer ID'].nunique())
print('Distinct products:', df['Product ID'].nunique())
print('Exact duplicate rows:', df.duplicated().sum())

## 2. Parse dates and validate ranges

Date parsing is explicit. Invalid dates are counted rather than silently ignored.

In [ ]:
for col in ['Order Date', 'Ship Date']:
    df[col] = pd.to_datetime(df[col], errors='coerce')

date_checks = pd.DataFrame({
    'min_date': [df['Order Date'].min(), df['Ship Date'].min()],
    'max_date': [df['Order Date'].max(), df['Ship Date'].max()],
    'invalid_count': [df['Order Date'].isna().sum(), df['Ship Date'].isna().sum()]
}, index=['Order Date', 'Ship Date'])
display(date_checks)

invalid_ship_before_order = (df['Ship Date'] < df['Order Date']).sum()
print('Ship date before order date:', invalid_ship_before_order)

## 3. Numeric and categorical validation

Check the numeric ranges and identify unexpected values before calculating KPIs.

In [ ]:
numeric_cols = ['Sales', 'Quantity', 'Discount', 'Profit']
display(df[numeric_cols].describe().T)

print('Negative sales rows:', (df['Sales'] < 0).sum())
print('Non-positive quantity rows:', (df['Quantity'] <= 0).sum())
print('Discount outside [0, 1]:', (~df['Discount'].between(0, 1)).sum())
print('Negative profit rows:', (df['Profit'] < 0).sum())

for col in ['Category', 'Sub-Category', 'Region', 'Segment', 'Ship Mode']:
    print(f'\n{col}: {df[col].nunique()} unique values')
    display(df[col].value_counts().rename('row_count').to_frame())

## 4. Baseline commercial profile

These are descriptive dataset totals. They are not causal findings and should be reconciled with SQL/Power BI later. AOV is calculated after aggregating line sales to order level.

In [ ]:
order_sales = df.groupby('Order ID', as_index=False)['Sales'].sum()
baseline = pd.Series({
    'sales_total': df['Sales'].sum(),
    'recorded_profit_total': df['Profit'].sum(),
    'units_total': df['Quantity'].sum(),
    'order_count': df['Order ID'].nunique(),
    'customer_count': df['Customer ID'].nunique(),
    'product_count': df['Product ID'].nunique(),
    'average_order_value': order_sales['Sales'].mean(),
    'negative_profit_line_count': (df['Profit'] < 0).sum(),
    'sales_on_negative_profit_lines': df.loc[df['Profit'] < 0, 'Sales'].sum()
})
display(baseline.to_frame('value').round(2))

## 5. First analysis cuts

Start with category and region summaries. These help guide deeper SQL analysis; they are not yet recommendations.

In [ ]:
category_summary = (df.groupby('Category', as_index=False)
    .agg(sales=('Sales', 'sum'), profit=('Profit', 'sum'), units=('Quantity', 'sum'),
         order_lines=('Row ID', 'count')))
category_summary['profit_margin_pct'] = np.where(
    category_summary['sales'].ne(0),
    category_summary['profit'] / category_summary['sales'] * 100,
    np.nan
)
display(category_summary.sort_values('sales', ascending=False).round(2))

region_summary = (df.groupby('Region', as_index=False)
    .agg(sales=('Sales', 'sum'), profit=('Profit', 'sum'), units=('Quantity', 'sum')))
region_summary['profit_margin_pct'] = np.where(
    region_summary['sales'].ne(0),
    region_summary['profit'] / region_summary['sales'] * 100,
    np.nan
)
display(region_summary.sort_values('sales', ascending=False).round(2))

## 6. Data limitations and next steps

- No store ID: use region/state/city as geographic cuts, not store-level comparisons.
- No inventory snapshots or stock-on-hand: inventory KPIs require a separate source.
- No explicit returns table in this file: do not infer returns from negative profit or other fields.
- Profit is a supplied field; its exact accounting definition should be confirmed from the dataset documentation.
- Before final reporting, reconcile all totals and document the source version/licence.

Next: inspect the separate inventory dataset and determine whether product/store/date keys can be validly integrated.